In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from ydata_profiling import ProfileReport
import pickle
import random
import itertools

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 50

data_folder = "../data/"
images_folder = "../data/images/"
pipelines_folder = "../pipelines/"
df_total = pd.read_csv(os.path.join(data_folder, 'items_phase_1.csv'))
df_train = pd.read_csv(os.path.join(data_folder, 'items_train.csv'))
df_task_1 = pd.read_csv(os.path.join(data_folder, 'task_1.csv'))

# Notebook `items_phase_1.csv`

In [2]:
# profile = ProfileReport(df_total, title="items_phase_1", explorative=True)
# profile.to_notebook_iframe()

In [3]:
print("Length of dataset:", len(df_total))

Length of dataset: 199835


In [4]:
df_total.sample(2)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo
89736,1018567,39990.00,230,['11'],NaN,"Calvin Klein Jeans hátizsák fekete, férfi, nag...",- Cipzáras zárás.\n- A belső zseb megkönnyíti ...,hu
52629,1052052,109.99,230,['1'],NaN,Lacoste oblekica,"Barva: črna , Koda proizvajalca: EF5473, Kroj:...",si


## Key takeaway
- missingy se musi resit u:
    - description (3%)
    - brandEditionTagId (99.8%) - to je asi target?
    - colorTagIdsString (3.1%)
---
# Notebook `task_1.csv`
- kazdy radek je jedna skupina se sloupci item - item4(to jsou id do items_phase_1.csv - itemId)
- Kazdy 

In [5]:
print("Length of dataset:", len(df_task_1))

Length of dataset: 15000


In [6]:
# profile = ProfileReport(df_task_1, title="task_1", explorative=True)
# profile.to_notebook_iframe()

In [7]:
df_task_1.head()

,item1,item2,item3,item4,item5
0,130622,292253,463442,483968,1253745
1,82627,388496,553738,638400,884327
2,46130,333489,644448,848154,1178149
3,150796,248537,742067,1206230,1280786
4,76610,196894,345145,620255,932761


---
# Dataset `items_train.csv`
- obsahuje toho min, vim ze ma stejny hodnoty v departmentIds


In [8]:
print("Total records:", len(df_train))
print("Total null values:\n", df_train.isnull().sum())

Total records: 928234
Total null values:
 itemId                    0
price                     0
colorTagIdsString     27834
departmentIds             0
brandEditionTagId    925518
title                     0
description           35473
geo                       0
label                     0
dtype: int64


In [9]:
df_train.sample(2)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo,label
231368,133556,20610.0,"778,1438011",['1'],NaN,Papucs Crocs,Papucs Crocs Crocs Furever Crush Clog 208446 Bézs,hu,8419
57334,335371,16.0,6466,['1'],NaN,Dámsky set Trendyol,Dámsky set Trendyol s prepracovanými detailmi ...,sk,127139


In [10]:
# profile = ProfileReport(df_train, title="task_1", explorative=True)
# profile.to_notebook_iframe()


---
# Vycisteni dat 
- prevod na spolecnou menu 
- normalizace meny v danem geo uzemi
- doplneni null hodnot
- null ve sloupci `colorTagIdString` muzu nahradit 0 
- `colorTagIdString` a `departmentIds` je potreba roztrhnout - obsahuji vice hodnot oddelenych carkou

## Tvorba preprocessing pipeline

In [11]:
df_total[df_total["brandEditionTagId"] == 0]

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo


In [12]:
df_train.columns

Index(['itemId', 'price', 'colorTagIdsString', 'departmentIds',
       'brandEditionTagId', 'title', 'description', 'geo', 'label'],
      dtype='object')

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from PriceGeoTransformer import PriceGeoTransformer
from DepartmentIdsTransformer import DepartmentIdsCleaner
import numpy as np


input_unknown_cols = ["description","title"]
input_unknown_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value="Unknown")),
])



# Pro textové IDčka (barvy) dáme prázdný string
imput_empty_string_cols = ['colorTagIdsString']
input_empty_string_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value="")),
])

imput_zero_cols = ["brandEditionTagId"]
input_zero_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
])


# prevadi na stejny format jako jsou barvy - cisla oddelena carkou 
department_features = ['departmentIds']
department_transformer = Pipeline(steps=[
    ('DepartmentIdsCleaner', DepartmentIdsCleaner())
])

# impute missing geo and convert back to pandas
imputer_step = ColumnTransformer(
    transformers=[
        ('geo_imp', SimpleImputer(strategy='constant', fill_value='<UNK>'), ['geo']),
        ('price_imp', SimpleImputer(strategy='median'), ['price'])
    ], 
    remainder='passthrough',
    verbose_feature_names_out=False 
).set_output(transform="pandas")

categorical_features = ['geo',"price"]
categorical_transformer = Pipeline(steps=[
    ("imputer", imputer_step),
    ('PriceGeoTransformer', PriceGeoTransformer())
])


# Combine preprocessing for numeric and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('zero', input_zero_transformer, imput_zero_cols),
        ('unknown', input_unknown_transformer, input_unknown_cols),
        ('geo', categorical_transformer, categorical_features),
        ('department', department_transformer, department_features)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

preprocessor.set_output(transform="pandas")



pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

X_train = df_train.drop(columns=['label'])
y_train = df_train['label']

X_train_transformed = pipeline.fit_transform(X_train)
df_train_transformed = X_train_transformed.copy()
df_train_transformed['label'] = y_train.values
df_total_transformed = pipeline.transform(df_total)

In [14]:
pickle.dump(pipeline, open(os.path.join(pipelines_folder, 'preprocessing_pipeline.pkl'), 'wb'))

## Příprava pro PyTorch dataset - Vocabulary pro transformaci kategorií
- PyTorch bude použit na vytvoření embeddingů sloupců s více kategorijema a na kategorické sloupce
- PyTorch umí totiž lépe tvořit řídké matice
- Vytvoříme si mappingy pro jednotlivé kategorie...

In [15]:
from GlamiDatasetVocabulary import GlamiVocabularyManager
vocab_manager = GlamiVocabularyManager() # vytvori vsechny potrebne mappingy kategorii do ciselnych hodnot
vocab_manager.fit(df_total_transformed)
vocab_manager.save()

## Tvorba embeddingů obrázků

In [16]:
import torch
from transformers import CLIPProcessor, CLIPVisionModel
from PIL import Image
import os
from tqdm import tqdm # Pro hezký progress bar

def create_and_save_clip_embeddings(image_dir, item_ids, save_path="clip_embeddings.pt", batch_size=64):
    # 1. Aktivace Apple M4 Pro (Metal Performance Shaders)
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("🚀 Běžíme na Apple M4 Pro (MPS backend)!")
    else:
        device = torch.device("cpu")
        print("⚠️ Běžíme na CPU.")

    # 2. Načtení CLIP modelu (potřebujeme jen tu část pro obrázky - VisionModel)
    model_id = "openai/clip-vit-base-patch32"
    print(f"Načítám model {model_id}...")
    processor = CLIPProcessor.from_pretrained(model_id)
    model = CLIPVisionModel.from_pretrained(model_id).to(device)
    model.eval() # Přepneme do módu vyhodnocování

    embeddings_dict = {}
    valid_ids = []
    images_to_process = []

    print("Připravuji obrázky...")
    # 3. Zpracování po dávkách (Batches) - klíčové pro rychlost na M4!
    for i in tqdm(range(0, len(item_ids), batch_size)):
        batch_ids = item_ids[i:i + batch_size]
        batch_images = []
        batch_valid_ids = []

        # Načteme obrázky v dávce
        for item_id in batch_ids:
            img_path = os.path.join(image_dir, f"{item_id}.jpg")
            try:
                img = Image.open(img_path).convert("RGB")
                batch_images.append(img)
                batch_valid_ids.append(str(item_id))
            except Exception:
                # Pokud obrázek fyzicky chybí, přeskočíme ho
                pass

        if not batch_images:
            continue

        # 4. Provedení samotné extrakce na čipu
        with torch.no_grad(): # Šetří paměť, nechceme počítat gradienty
            inputs = processor(images=batch_images, return_tensors="pt").to(device)
            outputs = model(**inputs)
            # pooler_output je ten finální "chytrý" 512-dimenzionální vektor
            image_embeds = outputs.pooler_output 

            # 5. Uložení do slovníku (přesuneme zpět na CPU, ať neplníme GPU paměť)
            image_embeds = image_embeds.cpu()
            for idx, i_id in enumerate(batch_valid_ids):
                embeddings_dict[i_id] = image_embeds[idx]

    # 6. Uložení na disk
    print(f"Ukládám {len(embeddings_dict)} embeddingů do {save_path}...")
    torch.save(embeddings_dict, save_path)
    print("Hotovo! 🎉")
    
    return embeddings_dict

# --- POUŽITÍ ---
# item_ids_list = df_train['itemId'].unique().tolist()
# my_embeddings = create_and_save_clip_embeddings("cesta/k/obrazkum", item_ids_list)

In [17]:
import os
import torch

os.environ["HF_HUB_OFFLINE"] = "1"

print("Připravuji CLIP embeddingy...")
item_ids_list = df_train_transformed['itemId'].unique().tolist()
embeddings_path = "clip_embeddings.pt"

# Chytrá kontrola: nebudeme to počítat znovu, pokud už to máme!
if os.path.exists(embeddings_path):
    print("Načítám hotové embeddingy z disku (blesková akce)...")
    clip_embeddings_dict = torch.load(embeddings_path)
else:
    print("Soubor nenalezen. Žhavím Apple M4 Pro a jdu počítat z obrázků...")
    # Tady zavoláme tu funkci z minula!
    clip_embeddings_dict = create_and_save_clip_embeddings(
        image_dir=images_folder, 
        item_ids=item_ids_list, 
        save_path=embeddings_path, 
        batch_size=128 # M4 Pro zvládne velké batche s přehledem
    )

print(f"Máme {len(clip_embeddings_dict)} vektorů obrázků!")

Připravuji CLIP embeddingy...
Načítám hotové embeddingy z disku (blesková akce)...
Máme 928233 vektorů obrázků!


## PyTorch Dataset


## V tomto stavu:
- Nejsou null hodnoty ve sloupcích:
    - colorTagIdString
    - description
    -  brandEditionTagId
- Barvy a deptId maji stejny format - cisla oddelena carkou
- chybi udelat OneHot encodingy kategorickejch - to udelame v PyTorchi
--- 


## Tvorba siamese datasetu
- dataset obsahuje páry z originálního datasetu teré mají jako label 1 pokud mají stejné labely uvitř páru, jinak 0
- create_balanced_pairs jich vytvori tolik aby byly vyvazene labely 

In [18]:
def create_balanced_pairs(df, item_col='itemId', group_col='label'):
    """
    Vygeneruje vyvážený dataset párů (50 % pozitivních, 50 % negativních).
    """
    print("Generuji pozitivní páry (duplikáty)...")
    positive_pairs = []
    
    # 1. Najdeme všechny produkty, které patří k sobě
    for _, group_df in df.groupby(group_col):
        items = group_df[item_col].tolist()
        if len(items) >= 2:
            # Vytvoříme všechny možné dvojice v rámci stejné skupiny
            positive_pairs.extend(list(itertools.combinations(items, 2)))
            
    num_positives = len(positive_pairs)
    print(f"Nalezeno {num_positives} pozitivních párů.")
    
    # 2. Rychlý slovník pro kontrolu negativních párů
    item_to_group = dict(zip(df[item_col], df[group_col]))
    all_items = df[item_col].tolist()
    
    print("Generuji negativní páry (odlišné produkty)...")
    negative_pairs = []
    
    # 3. Náhodně losujeme dvojice, dokud jich nemáme stejně jako pozitivních
    while len(negative_pairs) < num_positives:
        item1 = random.choice(all_items)
        item2 = random.choice(all_items)
        
        # Musí to být různé produkty a nesmí patřit do stejné skupiny!
        if item1 != item2 and item_to_group[item1] != item_to_group[item2]:
            negative_pairs.append((item1, item2))
            
    pos_df = pd.DataFrame(positive_pairs, columns=['item_id_1', 'item_id_2'])
    pos_df['is_duplicate'] = 1.0
    
    neg_df = pd.DataFrame(negative_pairs, columns=['item_id_1', 'item_id_2'])
    neg_df['is_duplicate'] = 0.0
    
    pairs_df = pd.concat([pos_df, neg_df]).sample(frac=1.0, random_state=42).reset_index(drop=True)
    
    print(f"Hotovo! Výsledný dataset má {len(pairs_df)} párů.")
    return pairs_df

In [19]:
from transformers import AutoTokenizer
from GlamiItemDataset import GlamiItemDataset
from GlamiSiameseDataset import GlamiSiameseDataset

# 1. Vygenerujeme vyvážené páry (1/0)
my_pairs_df = create_balanced_pairs(df_train_transformed)

# 2. Načteme tokenizer (doporučuji inicializovat ho takto mimo Dataset)
my_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 3. Vytvoříme "knihovnu" produktů a zapneme TURBO mód (předáme embeddings_dict)
base_dataset = GlamiItemDataset(
    df=df_train_transformed, 
    vocab_manager=vocab_manager, 
    tokenizer=my_tokenizer,
    embeddings_dict=clip_embeddings_dict, # <--- Magický přepínač na rychlé pole čísel
    max_len=128
)

# 4. Vytvoříme finální Siamský dataset připravený pro PyTorch DataLoader
siamese_dataset = GlamiSiameseDataset(item_dataset=base_dataset, pairs_df=my_pairs_df)

# Zkušební tisk:
vzorek = siamese_dataset[0]
print("--- TEST ÚSPĚŠNÝ ---")
print("Label vzorku:", vzorek['label'].item())
print("Tvar vektoru obrázku 1:", vzorek['item1']['image_embedding'].shape) # Bude [512]

Generuji pozitivní páry (duplikáty)...
Nalezeno 5546613 pozitivních párů.
Generuji negativní páry (odlišné produkty)...
Hotovo! Výsledný dataset má 11093226 párů.
--- TEST ÚSPĚŠNÝ ---
Label vzorku: 0.0
Tvar vektoru obrázku 1: torch.Size([768])


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GlamiItemEncoder(nn.Module):
    def __init__(self, num_geos, num_colors, num_depts, text_vocab_size, emb_dim=128):
        super().__init__()
        # 1. Tabulární data (Malé lineární a embedding vrstvy)
        self.geo_emb = nn.Embedding(num_embeddings=num_geos + 1, embedding_dim=16) # +1 pro <UNK>
        self.price_layer = nn.Linear(1, 16)
        self.color_layer = nn.Linear(num_colors, 32)
        self.dept_layer = nn.Linear(num_depts, 32)

        # 2. Text (Rychlý baseline: jednoduché učitelné embeddingy pro slova)
        self.text_emb = nn.Embedding(num_embeddings=text_vocab_size, embedding_dim=emb_dim, padding_idx=0)

        # 3. Obrázek (Tvá M4 Pro už udělala 768 vektor, jen ho trochu zmenšíme)
        self.image_layer = nn.Linear(768, 256)

        # 4. Spojení všeho dohromady (16 + 16 + 32 + 32 + 128 + 256 = 480)
        fusion_dim = 480
        self.fc1 = nn.Linear(fusion_dim, 512)
        self.fc2 = nn.Linear(512, 256) # Finální otisk produktu

    def forward(self, batch):
        # Tabulární data
        geo_x = self.geo_emb(batch['geo'])
        price_x = F.relu(self.price_layer(batch['price']))
        color_x = F.relu(self.color_layer(batch['colors']))
        dept_x = F.relu(self.dept_layer(batch['departments']))

        # Text (Mean Pooling: zprůměrujeme vektory slov podle attention masky)
        input_ids = batch['input_ids']
        mask = batch['attention_mask'].unsqueeze(-1)
        text_x = self.text_emb(input_ids)
        text_x = (text_x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

        # Obrázek
        image_x = F.relu(self.image_layer(batch['image_embedding']))

        # Spojení do jednoho super-vektoru
        x = torch.cat([geo_x, price_x, color_x, dept_x, text_x, image_x], dim=1)
        
        # Prohnání přes plně propojené vrstvy
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class GlamiSiameseNetwork(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        
        # Klasifikátor dostane na vstup: Vektor A, Vektor B, a jejich absolutní rozdíl a součin
        # To je 256 * 4 = 1024 čísel
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2), # Trochu prevence proti přeučení
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1) # Výstup je jediné číslo (tzv. logit)
        )

    def forward(self, item1, item2):
        u = self.encoder(item1)
        v = self.encoder(item2)

        # Extrakce vztahů mezi vektory (velmi osvědčený trik pro Siamese sítě!)
        diff = torch.abs(u - v)
        prod = u * v
        
        features = torch.cat([u, v, diff, prod], dim=1)
        logit = self.classifier(features)
        
        return logit.squeeze(-1) # Odstraníme zbytečnou dimenzi

In [21]:
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm

# 1. Příprava DataLoaderu (Naloží data do batche)
BATCH_SIZE = 128
train_loader = DataLoader(siamese_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 2. Inicializace modelu
# Potřebujeme zjistit velikosti slovníků
num_geos = len(vocab_manager.geo_dict)
num_colors = len(vocab_manager.color_dict)
num_depts = len(vocab_manager.dept_dict)
text_vocab_size = len(my_tokenizer)

encoder = GlamiItemEncoder(num_geos, num_colors, num_depts, text_vocab_size)
model = GlamiSiameseNetwork(encoder)

# Akcelerace na Macu!
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
print(f"Model běží na: {device}")

# 3. Optimalizátor a Loss funkce
# BCEWithLogitsLoss je nejlepší pro 1/0 klasifikaci (už v sobě obsahuje Sigmoid, takže je stabilnější)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

# 4. TRÉNOVACÍ SMYČKA
EPOCHS = 5 # Pro přes noc můžeš dát klidně 10 nebo 15

print("🚀 Startuji noční trénink...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct_preds = 0
    total_samples = 0
    
    # tqdm udělá krásný progress bar
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in progress_bar:
        # Přesun dat na grafiku (M4 Pro)
        item1 = {k: v.to(device) for k, v in batch['item1'].items() if isinstance(v, torch.Tensor)}
        item2 = {k: v.to(device) for k, v in batch['item2'].items() if isinstance(v, torch.Tensor)}
        labels = batch['label'].to(device)
        
        # Vynulování gradientů
        optimizer.zero_grad()
        
        # Dopředný průchod (Forward)
        logits = model(item1, item2)
        
        # Výpočet chyby
        loss = criterion(logits, labels)
        
        # Zpětný průchod a update vah (Backward)
        loss.backward()
        optimizer.step()
        
        # --- Statistiky pro logování ---
        total_loss += loss.item()
        
        # Převod logitů na pravděpodobnost (Sigmoid) a zaokrouhlení na 1/0
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        correct_preds += (preds == labels).sum().item()
        total_samples += labels.size(0)
        
        # Update progress baru o aktuální Accuracy
        current_acc = (correct_preds / total_samples) * 100
        progress_bar.set_postfix({'Loss': loss.item(), 'Acc': f"{current_acc:.1f}%"})

    epoch_acc = (correct_preds / total_samples) * 100
    epoch_loss = total_loss / len(train_loader)
    print(f"✅ Epoch {epoch+1} dokončena! Průměrný Loss: {epoch_loss:.4f} | Přesnost: {epoch_acc:.2f}%")

# Uložení modelu, ať o to ráno nepřijdeš!
torch.save(model.state_dict(), "glami_baseline_model.pth")
print("🎉 Model bezpečně uložen!")

Model běží na: mps
🚀 Startuji noční trénink...


Epoch 1/5:   3%|▎         | 2938/86666 [03:02<1:26:42, 16.10it/s, Loss=0.00808, Acc=99.1%] 


RuntimeError: stack expects each tensor to be equal size, but got [768] at entry 0 and [512] at entry 27